#### This file finds the scattering parameter and adds it to the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

#### Import Data

In [ ]:
clean_path = "../../data/spectral_library_clean.xlsx"
df_clean = pd.read_excel(clean_path)

experimental_path = "../../data/spectral_library_scattering_cyanobacteria.xlsx"
df_experimental = pd.read_excel(experimental_path)

same_sample_path = "../../data/spectral_library_scattering_diatomp_same_sample.xlsx"
df_same_sample = pd.read_excel(same_sample_path)

#### Compare Experimental Scattering Data with Unscattered Data

In [ ]:
wavelength_col = "Wavelength"
cyano_col = "Cyanobacteria_Synechosystis"

experimental_wavelengths = df_experimental[wavelength_col].values
experimental_spectrum = df_experimental[cyano_col].values
unscattered_wavelengths = df_clean[wavelength_col].values
unscattered_spectrum = df_clean[cyano_col].values


plt.figure(figsize=(8, 5))
plt.plot(experimental_wavelengths, experimental_spectrum, label="Experimental Scattering", alpha=0.8)
plt.plot(unscattered_wavelengths, unscattered_spectrum, label="Unscattered", alpha=0.8)
plt.xlabel("Wavelength (nm)")
plt.ylabel("Optical Density (OD)")
plt.title("Experimental Scattering vs Unscattered Absorption")
plt.legend(fontsize=7)
plt.show()

#### Fit scattering

In [ ]:
def fitting_function(wavelengths_nm, a, n, c):
    lam = wavelengths_nm.astype(float)
    lambda_ref = 550
    return a * (lam / lambda_ref) ** (-n) + c

mask = (experimental_wavelengths >= 770) & (experimental_wavelengths <= 800)
wl_fit = experimental_wavelengths[mask]
A_fit  = experimental_spectrum[mask]

p0 = (0.1, 1.0, 0.0)
popt, pcov = curve_fit(fitting_function, wl_fit, A_fit, p0=p0, maxfev = 200000)
a_hat, n_hat, c_hat = popt
print(f"Fitted parameters:  a={a_hat:.4f},  n={n_hat:.4f},  c={c_hat:.6f}")
perr = np.sqrt(np.diag(pcov))
print(f"Parameter uncertainties:  Δa={perr[0]:.4f},  Δn={perr[1]:.4f},  Δc={perr[2]:.6f}")

In [ ]:
wl_full = unscattered_wavelengths
A_full = unscattered_spectrum
fitted_scattering = fitting_function(wl_full, a=a_hat, n=n_hat, c=c_hat)

model_spectrum = A_full + fitted_scattering

plt.figure(figsize=(8, 5))
plt.plot(experimental_wavelengths, experimental_spectrum, 'r', label='Experimental Scattering', markersize=4)
plt.plot(unscattered_wavelengths, unscattered_spectrum, 'g', label='Unscattered Spectrum')
plt.plot(wl_full, model_spectrum, 'b', label='Model Spectrum')
plt.plot(wl_full, fitted_scattering, 'c--', label='Fitted Scattering', linewidth=2)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Optical Density (OD)')
plt.title('Fitted Scattering Spectrum for Cyanobacteria_Synechosystis')
plt.legend()
plt.grid()
plt.show()


#### Fit on scattering only

In [ ]:
scattering_only = pd.DataFrame(columns=["Wavelength", "Optical Density"])
mask = (experimental_wavelengths >= 350) & (experimental_wavelengths <= 800)
scattering_only["Wavelength"] = experimental_wavelengths[mask]
scattering_only["Optical Density"] = experimental_spectrum[mask] - unscattered_spectrum

scattering_only_wavelengths = scattering_only["Wavelength"]
scattering_only_optical_density = scattering_only["Optical Density"]

scattering_only["Optical Density"] = scattering_only["Optical Density"].replace(np.nan, 0)

# fit away from absorption peaks, over a wide range
mask = (
    ((scattering_only_wavelengths >= 450) & (scattering_only_wavelengths <= 600)) |
    ((scattering_only_wavelengths >= 720) & (scattering_only_wavelengths <= 800))
)
wl_fit = scattering_only_wavelengths[mask].values.astype(float)
A_fit  = scattering_only_optical_density[mask].values.astype(float)

p0 = [1.0, 1.0, 0.0]
# bounds the fitted parameters in some regime. We know it's Mie, so .3< n <2.5
bounds = ([0,   0.3,  -0.1],[5,   2.5,   0.3])

popt, pcov = curve_fit(
    fitting_function, wl_fit, A_fit,
    p0=p0, bounds=bounds, maxfev=200
)
a_hat, n_hat, c_hat = popt
print(f"Fitted parameters:  a={a_hat:.4f},  n={n_hat:.4f},  c={c_hat:.4f}")
perr = np.sqrt(np.diag(pcov))
print(f"Parameter uncertainties:  Δa={perr[0]:.4f},  Δn={perr[1]:.4f},  Δc={perr[2]:.4f}")

wl_full = unscattered_wavelengths
A_full = unscattered_spectrum
fitted_scattering = fitting_function(wl_full, a=a_hat, n=n_hat, c=c_hat)
model_scattering_only = A_full + fitted_scattering

plt.figure(figsize=(8, 5))
plt.plot(scattering_only_wavelengths, scattering_only_optical_density, 'y', label='Scattering Only', markersize=4)
plt.plot(wl_full, fitted_scattering, 'c--', label='Fitted Scattering', linewidth=2)
plt.plot(wl_full, model_scattering_only, 'g', label='Model with Scattering', linewidth=2)
plt.plot(experimental_wavelengths, experimental_spectrum, 'r', label='Experimental Measurement with Scattering', markersize=4)
plt.plot(unscattered_wavelengths, unscattered_spectrum, 'b', label='Unscattered Spectrum')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Optical Density (OD)')
plt.title('Scattering Only Spectrum for Cyanobacteria_Synechosystis')
plt.legend()
plt.grid()
plt.show()

In [ ]:
def linear_fitting_function(wavelengths_nm, a, b):
        return a * wavelengths_nm + b

scattering_only = pd.DataFrame(columns=["Wavelength", "Optical Density"])
mask = (experimental_wavelengths >= 350) & (experimental_wavelengths <= 800)
scattering_only["Wavelength"] = experimental_wavelengths[mask]
scattering_only["Optical Density"] = experimental_spectrum[mask] - unscattered_spectrum

scattering_only_wavelengths = scattering_only["Wavelength"]
scattering_only_optical_density = scattering_only["Optical Density"]

scattering_only["Optical Density"] = scattering_only["Optical Density"].replace(np.nan, 0)
mask = (scattering_only_wavelengths >= 400) & (scattering_only_wavelengths <= 800)
wl_fit = scattering_only_wavelengths[mask]
A_fit  = scattering_only_optical_density[mask]

p0 = (0.1, 1.0)
popt, pcov = curve_fit(linear_fitting_function, wl_fit, A_fit, p0=p0)
a_hat, b_hat = popt
print(f"Fitted parameters:  a={a_hat:.4f},  b={b_hat:.4f}")
perr = np.sqrt(np.diag(pcov))
print(f"Parameter uncertainties:  Δa={perr[0]:.4f},  Δb={perr[1]:.4f}")

wl_full = unscattered_wavelengths
A_full = unscattered_spectrum
fitted_scattering = linear_fitting_function(wl_full, a=a_hat, b=b_hat)
model_scattering_only = A_full + fitted_scattering

plt.figure(figsize=(8, 5))
plt.plot(scattering_only_wavelengths, scattering_only_optical_density, 'y', label='Scattering Only', markersize=4)
plt.plot(wl_full, fitted_scattering, 'c--', label='Fitted Scattering', linewidth=2)
plt.plot(wl_full, model_scattering_only, 'g', label='Model with Scattering', linewidth=2)
plt.plot(experimental_wavelengths, experimental_spectrum, 'r', label='Experimental Measurement with Scattering', markersize=4)
plt.plot(unscattered_wavelengths, unscattered_spectrum, 'b', label='Unscattered Spectrum')
plt.xlabel('Wavelength (nm)')
plt.ylabel('Optical Density (OD)')
plt.title('Scattering Only Spectrum for Cyanobacteria_Synechosystis')
plt.legend()
plt.grid()
plt.show()

#### Comparing Scattering and Without Scattering of Same Sample

In [ ]:
wavelength_diatom_col = "Wavelength (nm)"
scatter_col = "with_scattering"
unscatter_col = "without_scattering"

wavelengths = df_same_sample[wavelength_diatom_col].values
scattered_spectrum = df_same_sample[scatter_col].values
unscattered_spectrum = df_same_sample[unscatter_col].values

scatter_only_spectrum = scattered_spectrum - unscattered_spectrum
print(f"Scattering only spectrum (350 nm): {scatter_only_spectrum[-1]}")
print(f"Scattering only spectrum (750 nm): {scatter_only_spectrum[0]}")

mask_for_shift = (wavelengths >= 650) & (wavelengths <= 750)
unscattered_shift = unscattered_spectrum[mask_for_shift]
scattered_shift = scattered_spectrum[mask_for_shift]
wavelengths_shift = wavelengths[mask_for_shift]

wavelength_of_max_unscattered = wavelengths_shift[np.argmax(unscattered_shift)]
wavelength_of_max_scattered = wavelengths_shift[np.argmax(scattered_shift)]

shift_value = wavelength_of_max_unscattered - wavelength_of_max_scattered
print(f"Max unscattered (650-750 nm): {wavelength_of_max_unscattered:.4f}")
print(f"Max scattered (650-750 nm): {wavelength_of_max_scattered:.4f}")
print(f"Shift value to align peaks: {shift_value:.4f}")


plt.figure(figsize=(8, 5))
plt.plot(wavelengths, scattered_spectrum, label="Scattered", alpha=0.8)
plt.plot(wavelengths, unscattered_spectrum, label="Unscattered", alpha=0.8)
plt.plot(wavelengths, scatter_only_spectrum, label="Scattering Only", alpha=0.8)
plt.xlabel("Wavelength (nm)")
plt.ylabel("Optical Density (OD)")
plt.title("Scattering vs Unscattered Absorption of Diatom P")
plt.legend(fontsize=7)
plt.show()

In [ ]:
scattering_fit_diatom = pd.DataFrame(columns=["Wavelength", "Optical Density"])
mask = (wavelengths >= 350) & (wavelengths <= 750)
scattering_fit_diatom["Wavelength"] = wavelengths[mask]
scattering_fit_diatom["Optical Density"] = scatter_only_spectrum[mask]

scattering_fit_diatom_wavelengths = scattering_fit_diatom["Wavelength"]
scattering_fit_diatom_optical_density = scattering_fit_diatom["Optical Density"]

scattering_fit_diatom_optical_density = scattering_fit_diatom_optical_density.replace(np.nan, 0)

# fit away from absorption peaks, over a wide range
mask = (
    ((scattering_fit_diatom_wavelengths >= 450) & (scattering_fit_diatom_wavelengths <= 600)) |
    ((scattering_fit_diatom_wavelengths >= 720) & (scattering_fit_diatom_wavelengths <= 800))
)
wl_fit = scattering_fit_diatom_wavelengths[mask].values.astype(float)
A_fit  = scattering_fit_diatom_optical_density[mask].values.astype(float)

p0 = [1.0, 1.0, 0.0]
# bounds the fitted parameters in some regime. We know it's Mie, so .3< n <2.5
bounds = ([0,   0.3,  -0.1],[5,   2.5,   0.3])

popt, pcov = curve_fit(
    fitting_function, wl_fit, A_fit,
    p0=p0, bounds=bounds, maxfev=200
)
a_hat, n_hat, c_hat = popt
print(f"Fitted parameters:  a={a_hat:.4f},  n={n_hat:.4f},  c={c_hat:.4f}")
perr = np.sqrt(np.diag(pcov))
print(f"Parameter uncertainties:  Δa={perr[0]:.4f},  Δn={perr[1]:.4f},  Δc={perr[2]:.4f}")

wl_full = wavelengths
wl_full_shifted = wl_full - shift_value
A_full = unscattered_spectrum
fitted_scattering = fitting_function(wl_full, a=a_hat, n=n_hat, c=c_hat)
model_scattering_only = A_full + fitted_scattering


plt.figure(figsize=(8, 5))
plt.plot(wavelengths, scattered_spectrum, 'r', label='Experimental Measurement with Scattering', markersize=4)
plt.plot(wavelengths, unscattered_spectrum, 'm', label='Unscattered Spectrum')
plt.plot(scattering_fit_diatom_wavelengths, scattering_fit_diatom_optical_density, 'orange', label='Scattering Only', markersize=4)
plt.plot(wl_full, fitted_scattering, color='b', linestyle='--', label=f'Fitted Scattering n={n_hat:.2f}', linewidth=2)
# plt.plot(wl_full, model_scattering_only, 'm', label='Model with Scattering', linewidth=2)
plt.plot(wl_full_shifted, model_scattering_only, 'g-.', label='Model with Scattering', linewidth=2)
plt.xlabel('Wavelength (nm)')
plt.ylabel('Optical Density (OD)')
plt.title('Spectrum for Diatom Ptricornutum')
plt.legend(prop={'size': 6})
plt.grid()
plt.show()

peak_idx = np.argmax(scattered_spectrum[wavelengths > 600])
peak_wavelength = wavelengths[wavelengths > 600][peak_idx]
print(f"Experimental peak: {peak_wavelength:.1f} nm")

peak_idx_model = np.argmax(model_scattering_only[wl_full_shifted > 600])
peak_wavelength_model = wl_full_shifted[wl_full_shifted > 600][peak_idx_model]
print(f"Model peak: {peak_wavelength_model:.1f} nm")

print(f"Difference: {peak_wavelength - peak_wavelength_model:.1f} nm")

#### Species Size and Exponent Relation

In [ ]:
data = {
    "Diatom_Ptricornutum": {"diameter": 22, "exponent": 0, "fitted_exponent": 0.32},
    "Diatom_Csimplex": {"diameter": 24, "exponent": 0},
    "Chlamydomonas_Cpriscuii": {"diameter": 5},
    "Chlamydomonas_Creindhardtii": {"diameter": 16},
    "Dinoflagellate_Symbiodiniumsp": {"diameter": 13},
    "Dinoflagellate_Smicroadriaticum": {"diameter": 10},
    "Dinoflagellate_Dtrenchii": {"diameter": 10},
    "Dinoflagellate_Cgoreaui": {"diameter": 10},
    "Cyanobacteria_Synechosystis": {"diameter": 1, "exponent": 1.71, "fitted_exponent": 0.84},
    "VIR": {"diameter": 0.07, "exponent": 3.42},
    "BAC": {"diameter": 0.55, "exponent": 1.82},
}

short_labels = {
    "Diatom_Ptricornutum": "Ptri",
    "Diatom_Csimplex": "Csplx",
    "Chlamydomonas_Cpriscuii": "Cpri",
    "Chlamydomonas_Creindhardtii": "Crein",
    "Dinoflagellate_Symbiodiniumsp": "Sym",
    "Dinoflagellate_Smicroadriaticum": "Smic",
    "Dinoflagellate_Dtrenchii": "Dtre",
    "Dinoflagellate_Cgoreaui": "Cgor",
    "Cyanobacteria_Synechosystis": "Syn",
    "VIR": "VIR",
    "BAC": "BAC",
}

def fitting_exponents(diameter, a, b):
    return a * np.exp(b * diameter)


# 1) Build arrays for the two fits
# Fit 1: based on measured exponents (ignore 0 → treat as missing)
d_meas, n_meas, names_meas = [], [], []
for name, props in data.items():
    if "exponent" in props:
        d_meas.append(props["diameter"])
        n_meas.append(props["exponent"])
        names_meas.append(name)

d_meas = np.array(d_meas)
n_meas = np.array(n_meas)

# Fit 2: based on fitted_exponent values
d_fit, n_fit_vals, names_fit = [], [], []
for name, props in data.items():
    if "fitted_exponent" in props:
        d_fit.append(props["diameter"])
        n_fit_vals.append(props["fitted_exponent"])
        names_fit.append(name)

d_fit = np.array(d_fit)
n_fit_vals = np.array(n_fit_vals)

# 2) Perform the two exponential fits
# Reasonable initial guesses: a ~ n at smallest diameter, b negative
p0_meas = (n_meas.max(), -0.1)
popt_meas, pcov_meas = curve_fit(fitting_exponents, d_meas, n_meas, p0=p0_meas)

p0_fit = (n_fit_vals.max(), -0.05)
popt_fit, pcov_fit = curve_fit(fitting_exponents, d_fit, n_fit_vals, p0=p0_fit)

# 4) Plot both fitted curves and all species
all_diam = np.array([props["diameter"] for props in data.values()])
d_min, d_max = all_diam.min(), all_diam.max()
d_plot = np.linspace(d_min*0.7, d_max*1.3, 300)

plt.figure(figsize=(7,5))

# Measured exponents – big dots, no labels
plt.scatter(d_meas, n_meas, s=70, color="C0", edgecolor="k",
            zorder=3, label="Literature Exponents (n)")

# Fitted_exponent points – big dots, no labels
plt.scatter(d_fit, n_fit_vals, s=70, color="magenta", edgecolor="k",
            zorder=3, label="Fitted Exponents (n)")

# Fitted curves
plt.plot(d_plot, fitting_exponents(d_plot, *popt_meas),
         "C1-", label="Fit for Literature Exponents")
plt.plot(d_plot, fitting_exponents(d_plot, *popt_fit),
         "C2--", label="Fit for Fitted Exponents")

result = {}

# Predicted positions of all species on both curves (optional markers)
for name, props in data.items():
    d = props["diameter"]
    x = d
    y1 = fitting_exponents(d, *popt_meas)
    y2 = fitting_exponents(d, *popt_fit)
    # small grey markers to show where each species lies on each curve
    plt.plot(x, y1, "k.", ms=4)
    plt.plot(x, y2, "k.", ms=4)

    d = props["diameter"]
    n_pred_fit = fitting_exponents(d, *popt_fit)

    short_name = short_labels.get(name, name)

    # store diameter + last column in one dict
    result[short_name] = {
        "name": name,
        "diameter": d,
        "n_fit_observed": n_pred_fit
    }

exponent_df = pd.DataFrame(result).T
exponent_df = exponent_df.reset_index(drop=True)
print(exponent_df)

plt.xlabel("Diameter [µm]")
plt.ylabel("Scattering exponent (n)")
plt.grid(True, ls=":", alpha=0.4)
plt.legend(fontsize=8)
plt.tight_layout()
plt.title("Scattering Exponent (n) vs Diameter for Various Species")
plt.show()

#### Add scattering to all species

In [ ]:
def shifted_fitting_function(wavelengths_nm, n, a=1, c=0, shift=0):
    lam = wavelengths_nm.astype(float) + shift
    lambda_ref = 550
    return a * (lam / lambda_ref) ** (-n) + c


wavelength_col = "Wavelength"
wavelengths = df_clean[wavelength_col].values
wavelengths_shifted = wavelengths - shift_value

df_scattering = pd.DataFrame({wavelength_col: wavelengths_shifted})

species_list = [
    "Diatom_Ptricornutum",
    "Diatom_Csimplex",
    "Chlamydomonas_Cpriscuii",
    "Chlamydomonas_Creindhardtii",
    "Dinoflagellate_Symbiodiniumsp",
    "Dinoflagellate_Smicroadriaticum",
    "Dinoflagellate_Dtrenchii",
    "Dinoflagellate_Cgoreaui",
    "Cyanobacteria_Synechosystis",
]

for species in species_list:
    unscattered = df_clean[species]

    n_hat = exponent_df.loc[
        exponent_df["name"] == species, "n_fit_observed"
    ].iloc[0]

    if species == "Diatom_Ptricornutum":
        a_hat = a_hat
        c_hat = c_hat
    else:
        a_hat = 1.0
        c_hat = 0.0

    print(
        f"Processing {species}: using n={n_hat:.2f}, a={a_hat:.2f}, "
        f"c={c_hat:.2f}, shift={shift_value:.2f} nm"
    )

    scat = shifted_fitting_function(
        wavelengths,
        n=n_hat,
        a=a_hat,
        c=c_hat,
        shift=shift_value,
    )

    with_scattering = unscattered + scat

    df_scattering[species] = with_scattering

    plt.figure(figsize=(8, 5))
    plt.plot(wavelengths, unscattered, label=f"Unscattered {species}", alpha=0.8)
    plt.plot(wavelengths, scat, label="Fitted Scattering Component", alpha=0.8)
    plt.plot(wavelengths_shifted, with_scattering,
             label=f"{species} with Scattering", alpha=0.8)
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Optical Density (OD)")
    plt.title(f"{species}: Unscattered vs With Scattering")
    plt.legend(fontsize=7)
    plt.grid()
    plt.show()

df_clean = df_clean.drop("Wavelength_shifted", axis=1)
df_clean.to_excel("../../data/spectral_library_clean.xlsx", index=False)

print(df_scattering)
df_scattering.to_excel("../../data/spectral_library_with_scattering.xlsx",
                       index=False)